In [ ]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

import datetime

import pandas as pd

from pandas import ExcelWriter

from selenium import webdriver

from selenium.webdriver.common.by import By

from time import sleep

import os



# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'BM BMA' ## change to current controller name



print(f"Running {regulatorName} Web Scraping Tool v.1.1")

now=datetime.datetime.now()

filename= '{} SQL Ready data {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

#scriptfolder = f"C:\\Users\\siewekoa\\OneDrive - moodys.com\\Desktop\\My_data\\Project_work\\scripts_regulator\\{regulatorName}" ## to comment for the production environment

scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)



# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder,

         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert

         }

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

# regdict = {

#             "BM BMA 1": "Banking",

#             "BM BMA 2": "Investment Business", 

# 		    "BM BMA 3": "Trust Business", 

#             "BM BMA 4": "Insurance",

#             "BM BMA 5": "Corporate Service Provider",

#             "BM BMA 6": "Fund Administrator",

#             "BM BMA 7": "Money Service Business",

#             "BM BMA 8": "Digital Assets Business",

#             }



regdict = {

    'Banking'                   :'1',

    'Investment Business'       :'2',

    'Trust Business'            :'3',

    'Insurance'                 :'4',

    'Corporate Service Provider':'5',

    'Fund Administrator'        :'6',

    'Money Service Business'    :'7',

    'Digital Assets Business'   :'8',

    

    'Investment Business: Registered Persons'   : '2',

    'Investment Business: Licensed Entities'    : '2',

    'Credit Union'  : '0',

    'Non-Licensed Persons'  : '0',

    }





sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

         'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

         'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

         'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

         'Phone - Mother company': [], 'Check': []}



location = ['Incorporated in Hong Kong', 'Incorporated outside Hong Kong', 'Offices in Hong Kong']

processdate = now.strftime('%Y-%m-%d')



# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict



# %%

#------------------------------------------------ Begin_Main ----------------------------------------

driver.get('https://www.bma.bm/regulated-entities')

element = driver.find_element(By.ID,"loadmorebtn")

driver.execute_script("arguments[0].click();", element)

print(f"[INFO] : click on 'VIEW ALL +' button ")

sleep(2)

tbl = driver.find_element(By.CLASS_NAME,'table-responsive').get_attribute('outerHTML')

df  = pd.read_html(tbl)

res = df[0]

res['Phone'] = res['Contact Information'].str.extract(r"(\W*?Phone:\W*(\d{3}[(-)]?[\W]?[\r\n]?)+)")[0]

res['Phone'].replace('Phone:', '', regex=True, inplace=True)

res['Email'] = res['Contact Information'].str.extract(r"(\W*?Email:\W*\S+@\S+)")[0]

res['Email'].replace('Email:','', regex=True, inplace=True)

res['Fax'] = res['Contact Information'].str.extract(r"(\W*?Fax:\W*(\d{3}[(-)]?[\W]?[\r\n]?)+)")[0]

res['Fax'].replace('Fax:','', regex=True, inplace=True)

res['Web'] = res['Contact Information'].str.extract(r"(\W*?Website:\W*\S+\W?\S+)")[0]

res['Web'].replace('Web:','',regex=True,inplace=True)

res['Licence Information'] = res['License/Registration Information'].str.split(':').str[1]

res['ListCode'] = [regdict[res['Sector'][i]] for i in range(len(res))]



print(f"[INFO] : table  containe = {res.shape}")

for i in range(len(res)):

    if res['ListCode'].values[i] != '0' :

        

        if res['Address'][i].endswith('Bermuda')==True:

            res['Country'] = 'Bermuda'

            res['Address'][i] = res['Address'][i].replace('Bermuda','')

        else:

            res['Country'] = ''



        sqldict['Name'].append(res['Company Name'].values[i])

        sqldict['Website'].append(res['Web'].values[i])

        sqldict['Phone'].append(res['Phone'].values[i])

        sqldict['Fax'].append(res['Fax'].values[i])

        sqldict['Email'].append(res['Email'].values[i])

        sqldict['Address_1'].append( str(res['Address'].values[i]).replace('Principal Address', ''))

        sqldict['Typology'].append(res['Sector'].values[i])

        # sqldict['License_Type'].append(res['Licence Information'].values[i])

        sqldict["Cntry"].append(res['Country'].values[i])   

        sqldict["RegCtry"].append("BM")                

        sqldict["RegCode"].append("BMA")

        sqldict['ListProcessDate'].append(processdate)

        sqldict["ListCode"].append(res['ListCode'].values[i])  

        sqldict["RegulationType"].append('Supervised')

        sqldict["RegulationDate"].append(str(res['Date'].values[i]).replace('---', ''))

sqldict = bourange_same_length_array(sqldict)



# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)


    